# 02 · First cable exercise: nodes and segments
**CAS Computational Form-finding · Block Research Group · ETH Zürich**

Start with the simplest cable model: **5 nodes**, **4 straight segments**, **2 fixed end nodes** and **3 free nodes**. Apply the same downward point load at every free node, then find the hanging shape.

Choose **Run → Run All Cells**, or run each cell with **Shift+Enter**. No installation is needed on your computer; the first browser run downloads Python and packages.


In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["compas==2.15.1", "matplotlib"])
else:
    # Desktop Jupyter / Google Colab: install missing packages in this kernel.
    import importlib.util
    import subprocess
    if any(importlib.util.find_spec(name) is None for name in ["compas", "matplotlib"]):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "compas==2.15.1", "matplotlib"])

import numpy as np
import matplotlib.pyplot as plt
from compas.geometry import Point, Line


## 1. Define the nodes and the cable segments
A node is a point `(x, y, z)` in metres. A segment connects two node numbers. Nodes **0** and **4** are fixed supports; nodes **1**, **2** and **3** can move. All nodes start at `z = 0`.


In [ ]:
nodes = [
    Point(0, 0, 0),  # node 0: left support
    Point(1, 0, 0),  # node 1: free
    Point(2, 0, 0),  # node 2: free
    Point(3, 0, 0),  # node 3: free
    Point(4, 0, 0),  # node 4: right support
]
edges = [(0, 1), (1, 2), (2, 3), (3, 4)]
fixed = [0, 4]
free = [1, 2, 3]

P = 1.0  # downward load at EACH free node [kN]
q = 2.0  # force density (tension / segment length) [kN/m]

for i, point in enumerate(nodes):
    print(f"Node {i}: {list(point)} — {'fixed' if i in fixed else 'free'}")
print("Segments:", edges)


## 2. Draw the numbered nodes
Blue squares are supports. Pink dots are free nodes. Green arrows show the downward nodal loads. The grey line is the starting geometry, before equilibrium is solved.


In [ ]:
def draw_cable(nodes, title, color="#dc4b8d"):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for a, b in edges:
        ax.plot([nodes[a].x, nodes[b].x], [nodes[a].z, nodes[b].z],
                color=color, linewidth=2)
    for i, point in enumerate(nodes):
        ax.scatter(point.x, point.z, s=65, zorder=3,
                   marker="s" if i in fixed else "o",
                   color="#141b3d" if i in fixed else "#dc4b8d")
        ax.annotate(str(i), (point.x, point.z), xytext=(0, 12),
                    textcoords="offset points", ha="center")
        if i in free:
            ax.annotate("", xy=(point.x, point.z - 0.4 * P),
                        xytext=(point.x, point.z),
                        arrowprops={"arrowstyle": "->", "color": "#48884b", "lw": 2})
    ax.set(xlabel="x [m]", ylabel="z [m]", title=title,
           xlim=(-0.4, 4.4), ylim=(min(p.z for p in nodes) - 0.4 * P - 0.3, 0.55))
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

draw_cable(nodes, "Starting geometry · 5 nodes, 4 segments", color="#aaaabb")


## 3. Find the hanging shape
At each free node, the cable forces and the applied load must add up to zero. With the same positive force density `q` in all four segments, vertical equilibrium gives three equations:

```
2q z1 − q z2         = −P
−q z1 + 2q z2 − q z3 = −P
        −q z2 + 2q z3 = −P
```

The supports stay at `z0 = z4 = 0`. Equal force densities and no horizontal loads keep the x coordinates evenly spaced. NumPy solves these three equations for the free nodes' heights.

This is a simple **force density** example: we prescribe `q = tension / length`, rather than the cable's total length or stiffness. Straight segments are appropriate for point loads applied at the nodes.


In [ ]:
A = q * np.array([
    [ 2, -1,  0],
    [-1,  2, -1],
    [ 0, -1,  2],
], dtype=float)
loads_z = np.full(3, -P)
z_free = np.linalg.solve(A, loads_z)

for i, z in zip(free, z_free):
    nodes[i].z = float(z)

for i, point in enumerate(nodes):
    print(f"Node {i}: x = {point.x:g} m, z = {point.z:g} m")
print(f"Midspan sag: {-nodes[2].z:g} m")


In [ ]:
draw_cable(nodes, "Equilibrium cable · downward loads at the free nodes")

for a, b in edges:
    segment = Line(nodes[a], nodes[b])
    print(f"Segment {a}–{b}: length {segment.length:.3f} m, "
          f"tension {q * segment.length:.3f} kN")

# Check force balance at every free node.
xyz = np.array([list(point) for point in nodes])
for i in free:
    cable_force = q * (xyz[i - 1] - xyz[i]) + q * (xyz[i + 1] - xyz[i])
    residual = cable_force + np.array([0.0, 0.0, -P])
    assert np.allclose(residual, 0.0), f"Node {i} is not in equilibrium"
print("All three free nodes are in equilibrium.")


## 4. Try it yourself
1. Before running, predict which node will be lowest.
2. Double `P` from 1 to 2 kN. What happens to the sag?
3. Restore `P = 1`, then double `q` from 2 to 4 kN/m. What happens now?

After changing `P` or `q` in section 1, choose **Run → Run All Cells** again.

**Check your starting result:** the free-node heights are **−0.75, −1.00, −0.75 m**. The middle node is **1 m** below the supports.

Use `/notebook cable` in the BRG terminal for instructions. Paste a cell or its error there when asking for help. Save in your browser with **Ctrl+S** (or **⌘S**) and download your work through **File → Save and Export Notebook As → Notebook**.
